# 0.94656 by reading the public split — two exact AUC identities

Everything at the top of this Code tab sits on one file,
[@jazivxt's](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) `submission_latest_best.csv`
(0.94651). This notebook moves that file to **0.94656** without a new model. It uses two identities
that turn a submission into a measurement of the public split.

**Read this before forking.** What is exploited here is the sampling noise of the public 20%. It moves
the public score and, by construction, does nothing for the private one. Treat it as a way to understand
what a one-digit public gap is made of, not as a model.

## Identity 1 — a pair of submissions reads a group's labels

Put a group of rows G at the very top of a file, then at the very bottom. Whatever G's original
position,

$$\text{score(top)} - \text{score(bottom)} = \frac{k}{n_1} - \frac{b}{n_0}$$

where $k$, $b$ are the public positives and negatives inside G and $n_1$, $n_0$ the public totals.
I checked it on the 3,575 test rows of the three cells that have no buyer at all in training
(income 31,004–41,970; commute ≥ 83 km; income 30,000 without subsidy and with low concern or
medium/high anxiety):

| | value |
|---|---|
| predicted if those rows hold no positive | −0.01511 |
| measured, 0.93149 − 0.94651 | −0.01502 |
| implied positives in the 3,575 rows | about 4 (base rate would give 624) |

One positive is worth about two units of the last displayed digit, so the reading resolves to a
fraction of a row. A one-digit public gap is half a positive.

## Identity 2 — the gain from moving a block

Lift block G above block C and the public score changes by

$$\Delta = \frac{M_G \, M_C \, (\text{rate}_G - \text{rate}_C)}{n_1 n_0}$$

with $M$ the public rows in each block. A calibrated file orders blocks by predicted rate, so
swapping two adjacent blocks normally costs a little. But the public split is a random fifth of test,
so in some places the lower block's *public* rate is actually higher. Swap there and the score rises.

In [ ]:
import glob, hashlib, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"{ROOT}/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

path = find_one("zoom-zoom-baseline/submission_latest_best.csv", "submission_latest_best.csv")
EXPECTED = "1f40cec214646ba50eac6d5307d41c6aca82fe7acc38e2501399980c62d4bbe7"   # jazivxt submission 56267408
sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
if sha != EXPECTED:
    print("WARNING: the anchor file has changed since this notebook was measured.")
    print("The bands below were measured on submission 56267408; on a new anchor they move different rows")
    print("and the score will not be 0.94656.")
anchor = pd.read_csv(path).sort_values(ID)
ids, N = anchor[ID].to_numpy(), len(anchor)
r0 = rankdata(anchor[TARGET].to_numpy(dtype=float), method="average") / N
print(f"anchor rows {N:,} | sha256 matches the measured anchor: {sha == EXPECTED}")

## Which bands, and what each one did

For each band of the ranking I estimated, from a model's out-of-fold predictions, how much swapping
its two halves costs on average (the systematic term) and how much the public sampling noise can move
it (the noise term). Bands below the median carry almost no positives and can't move anything; the
useful ones sit between the 20th and 90th percentile. Each swap was submitted once, on its own, on top
of the current best file — so every submission was both a measurement and a candidate, and a failure
cost nothing but the slot.

**Round 1** (on the anchor, 0.94651):

| band (rank percentile) | predicted mean ± noise (units) | public LB | change |
|---|---|---|---|
| **0.54 – 0.58** | −0.61 ± 1.87 | **0.94653** | **+2** |
| 0.20 – 0.26 | −0.08 ± 0.80 | 0.94651 | 0 |
| 0.58 – 0.62 | −2.06 ± 2.29 | 0.94651 | 0 |
| 0.46 – 0.52 | −1.58 ± 2.59 | 0.94650 | −1 |
| 0.32 – 0.40 | −1.05 ± 2.47 | 0.94649 | −2 |
| 0.42 – 0.46 | −0.38 ± 1.12 | 0.94649 | −2 |
| first three kept together | | 0.94653 | +2 |

**Round 2** (on the 0.94653 file):

| band | predicted mean ± noise | public LB | change |
|---|---|---|---|
| **0.47 – 0.53** | −1.28 ± 2.68 | **0.94656** | **+3** |
| 0.88 – 0.90 | −1.42 ± 1.93 | 0.94653 | 0 |
| 0.40 – 0.46 | −0.64 ± 2.03 | 0.94652 | −1 |
| 0.67 – 0.69 | −0.68 ± 1.26 | 0.94652 | −1 |
| 0.64 – 0.67 | −1.66 ± 2.02 | 0.94651 | −2 |
| 0.70 – 0.72 | −0.99 ± 1.51 | 0.94651 | −2 |
| 0.31 – 0.39 | −0.91 ± 2.39 | 0.94650 | −3 |
| 0.77 – 0.79 | −1.32 ± 1.93 | 0.94649 | −4 |
| 0.79 – 0.81, on top of 0.94656 | −1.84 ± 1.99 | 0.94652 | −4 |

Two hits in fifteen, spread around a mean just below zero — which is what the predicted noise said to
expect. The hits are not skill; they are the places where this particular public sample happens to
disagree with the model's ordering.

In [ ]:
KEPT = [(0.54, 0.04), (0.20, 0.06), (0.58, 0.04), (0.47, 0.06)]   # (start, width): halves swapped

def swap_bands(r, bands):
    rr = r.copy()
    for lo, w in bands:
        half = w / 2
        lower = (r >= lo) & (r < lo + half)
        upper = (r >= lo + half) & (r < lo + w)
        rr[lower] += half
        rr[upper] -= half
    return rankdata(rr, method="ordinal") / len(rr)

final = swap_bands(r0, KEPT)
moved = (np.argsort(np.argsort(final)) != np.argsort(np.argsort(r0))).sum()
print(f"rows whose rank changed: {moved:,} | spearman vs anchor {spearmanr(final, r0).statistic:.6f}")
pd.DataFrame({ID: ids, TARGET: final}).to_csv("submission.csv", index=False)
print("wrote submission.csv")

## What this does and does not tell you

It tells you that the leaderboard's top digit is a statement about a few hundred specific public rows,
and that those rows can be located with a handful of submissions. Everyone above 0.9465 on the public
board is, to some degree, standing on that kind of information — whether they meant to or not.

It does not tell you anything about the private split. The swaps were chosen *because* the public
sample disagreed with the model; on the private rows the same swaps cost, on average, exactly the
systematic term in the tables above. If you select a final submission, pair this with one built from
out-of-fold predictions (for instance
[this one](https://www.kaggle.com/code/megayak/s6e9-0-94645-from-oof-files-only)).

---
Credit: [@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) for the anchor file.
The identities, the measurements and the swaps are mine. If this changed how you read a one-digit
public gap, an upvote helps.